# Thesis pipeline on Google Colab — with live progress monitoring

This notebook runs `code/run_all.py` (the approved entry point) on a Colab Pro+ High-RAM
runtime with four workers. It changes **nothing** in the statistical computations: it only
installs the pinned environment, starts the run, and shows the progress monitor's output.

What you get while the run is going:

* the console log, live, in this notebook;
* `run_logs/<namespace>/<run_id>/` on the VM: `execution.log`, `events.jsonl`,
  `progress.json`;
* the same three files mirrored to Google Drive every two minutes (`--log-mirror`), so
  they survive if the runtime is terminated.

Run the cells in order. Cell 6 starts the run **in the background**, so a browser
disconnect does not stop it; cell 7 can be re-run at any time to reattach to the live log.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_LOGS = '/content/drive/MyDrive/thesis_run_logs'      # mirrored run logs
DRIVE_RESULTS = '/content/drive/MyDrive/thesis_results'    # copy of output/ at the end
import os
os.makedirs(DRIVE_LOGS, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

## 2. Get the repository

Set `BRANCH` to the branch you want to run. Private repositories need a GitHub token in
Colab's *Secrets* panel (key `GITHUB_TOKEN`).

In [ ]:
REPO = 'charlijomedia-dotcom/academia7'
BRANCH = 'claude/progress-monitoring'

import os, subprocess
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/academia7'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, '/content/academia7'], check=True)
else:
    subprocess.run(['git', '-C', '/content/academia7', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/academia7', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/academia7', 'pull', '--ff-only'], check=True)
%cd /content/academia7/my-thesis
!git log --oneline -1

## 3. Install the pinned dependencies

Colab's own packages are left alone; the locked set is installed into the runtime. The
pipeline was validated on Python 3.11.

In [ ]:
!python --version
!pip install -q -r code/requirements.lock.txt

## 4. Verify the frozen data (seconds; no network, no analysis)

In [ ]:
!python -u code/run_all.py --data-only

## 5. Optional: reduced smoke run first (about 30 minutes)

Shows the monitor's output on a short run. Its results go to `smoke_output/` and are
never thesis-facing.

In [ ]:
# !python -u code/run_all.py --smoke-test --log-mirror "$DRIVE_LOGS"

## 6. Start the full run in the background

`python -u` and the entry point's line buffering keep the log current. The process is
started with `nohup`, so it keeps running if this browser tab disconnects. The console
output also goes to `/content/colab_console.log` — outside the clone, because the
clean-tree safeguard refuses to start a baseline run beside an untracked file.

The clean-tree safeguard requires a committed working tree: do not edit files in the
clone before this cell.

In [ ]:
import subprocess, os
cmd = (f'nohup python -u code/run_all.py --log-mirror "{DRIVE_LOGS}" '
       f'> /content/colab_console.log 2>&1 & echo $!')
pid = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
open('/content/colab_run.pid', 'w').write(pid)
print('started run_all.py, pid', pid)

## 7. Watch the live log (re-run any time to reattach)

Prints new console lines as they appear and, every five minutes, the key fields of the
atomically written `progress.json`. Interrupting this cell (■) stops the **watching**, not
the run.

In [ ]:
import json, os, time, pathlib

def latest_snapshot():
    root = pathlib.Path('run_logs/baseline')
    pointer = root / 'LATEST'
    if not pointer.exists():
        return None
    path = root / pointer.read_text().strip() / 'progress.json'
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return None

def running():
    try:
        os.kill(int(open('/content/colab_run.pid').read()), 0)
        return True
    except Exception:
        return False

offset, last_snapshot = 0, 0.0
while True:
    with open('/content/colab_console.log', 'r', errors='replace') as handle:
        handle.seek(offset)
        chunk = handle.read()
        offset = handle.tell()
    if chunk:
        print(chunk, end='', flush=True)
    if time.time() - last_snapshot > 300:
        last_snapshot = time.time()
        snap = latest_snapshot()
        if snap:
            stage = snap.get('current_stage') or {}
            print(f"\n[snapshot {snap['updated_utc']}] status={snap['status']} "
                  f"stage={stage.get('number')}/10 {stage.get('name')} "
                  f"remaining origins={snap['origins']['all']['remaining']}", flush=True)
    if not running():
        print('\nrun_all.py has exited; see the end of the log above.')
        break
    time.sleep(5)

## 8. After the run: copy the results to Drive

`output/` holds the thesis-facing results; `run_logs/` is already mirrored.

In [ ]:
!tail -n 40 /content/colab_console.log
!cp -r output "$DRIVE_RESULTS/output_$(date -u +%Y%m%dT%H%M%SZ)"
!ls "$DRIVE_RESULTS"

## If the runtime was terminated

Open `thesis_run_logs/baseline/<run_id>/progress.json` on Drive. It records the stage,
the active chains with their series, model, forecast origin and how long the current fit
had been running, the last checkpoint, and the last completed computation — i.e. what
was running when the VM stopped. `events.jsonl` has the full event history.